<a href="https://colab.research.google.com/github/Addychauhan/health_data_analysis/blob/main/Data_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Python & Data Wrangling
**Assignment:** Clean a messy dataset in Pandas — handle missing values, filter rows, create new columns.

#Importing the Required Libraries

In [ ]:
import numpy as np
import pandas as pd

#Loading the Dataset

In [ ]:
df=pd.read_csv('/content/data.csv')

In [ ]:
df

#Basic Information

In [ ]:
print("The shape of the Dataset is: ", df.shape)

#Checking for Missing or Null Values

In [ ]:
print("The number of missing values is: ", df.isnull().sum())

3 missing values present in the dataset

#Data Types of column

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

#Data Cleaning

##Number of missing values

In [ ]:
print("The number of missing values is: ", df.isnull().sum())

##Checking Duplicate values

In [ ]:
print("The number of Duplicate values is: ",df.duplicated().sum())

In [ ]:
df[df.duplicated(keep=False)]

#Identifying the Problems

| # | Problem | Where |
|---|----------|------|
|1| Dates are stored as text, wrapped in quotes (`'2020/12/01'`)|`Date` column|
|2| One date is in completely different format (`20201226`, no quotes/slashes)| Row 26 |
|3| One date is missing entirely |Row 22|
|4| Two `Calories` values are missing | Rows 18 and 28 |
|5| One `Duration` is `450`-clearly a typo for `45` |Row 7|
|6| `Pulse=130' But the `Maxpulse=101', which physically impossible, beacuse pulse can't exceed your max pulse |Row 23|

#Clean the data

##Fix the Date Column

In [ ]:
df["Date"] = df["Date"].astype(str).str.replace("'", "", regex=False)

In [ ]:
# Convert to real datetime values
df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')

In [ ]:
df

##Handle Missing Date Value

In [ ]:
print("Rows before:", len(df))
df = df.dropna(subset=['Date'])
print("Rows after: ", len(df))

In [ ]:
df

##Remove Duplicate Rows

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))

##Handle Missing Calories

In [ ]:
print("Missing Calories:", df['Calories'].isnull().sum())

median_calories = df['Calories'].median()
print("Median Calories:", median_calories)

In [ ]:
#Filling Missing Calories with Median

df['Calories'] = df['Calories'].fillna(median_calories)

print("Missing Calories after filling:", df['Calories'].isnull().sum())

##Fix the Duration Outlier

In [ ]:
print("Duration values before:", sorted(df['Duration'].unique()))

In [ ]:
#Visualize the outlier

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,3))
sns.boxplot(x=df['Duration'])
plt.title("Duration- 450 is the outlier obviously")
plt.show()

In [ ]:
#Correct the typo
df.loc[df['Duration']>300, 'Duration']=45
print("Duration values after:", sorted(df['Duration'].unique()))

##Investigate the Pulse Anomaly
Row 23 records `Pulse=130` with `maxpulse=101`. Pulse can't be higher than the maximum pulse, so one of these two numbers is wrong--- Most likely they were swapped during data entry

In [ ]:
anomaly=df[df['Pulse']>df['Maxpulse']]
anomaly

In [ ]:
#Swap these two values back into a sensible order
swap_anomaly=df['Pulse']>df['Maxpulse']
df.loc[swap_anomaly,['Pulse','Maxpulse']]=df.loc[swap_anomaly,['Maxpulse','Pulse']].values
print('Remaining Anomalies: ', (df['Pulse']>df['Maxpulse']).sum())
df.loc[swap_anomaly]

#Filter Rows
Now since the data is clean, we can filter it reliably.

In [ ]:
#Long workout
long_workouts=df[df['Duration']>=60]
print('Long workout of 60+ minutes: ', len(long_workouts))
long_workouts

In [ ]:
#High-intensity sessions: long and high calorie burn
high_intensity_session=df[(df['Duration']>=60) & (df['Calories']>350)]
print('Long, high-calorie workouts: ', len(high_intensity_session))
high_intensity_session

In [ ]:
#Filter by date range
late_dec=df[df['Date']>='	2020-12-20']
print('Workouts from Dec 20 onwards: ', len(late_dec))
late_dec

#Create New Columns

In [ ]:
#Calorie burned per minute
df['Calories_per_minute']=(df['Calories']/df['Duration']).round(2)
df

In [ ]:
#How hard was the session as a % of max pulse
df['Intensity_percent']=((df['Pulse']/df['Maxpulse'])*100).round(1)
df

In [ ]:
#Extract the weekday out of the date
df['Day_of_week']=df['Date'].dt.day_name()
df

In [ ]:
#Categorize each workout into a simple category
df['Workout_type']=pd.cut(
    df['Duration'],
    bins=[0,40,50,80],
    labels=['Short', 'Medium', 'Long']
)
df

#Cleaned Dataset

In [ ]:
print('Final Shape: ', df.shape)
print('\nMissing Values: ')
print(df.isnull().sum())
print("nDuplicate Records: ")
print(df.duplicated().sum())
print("nData Types: ")
print(df.dtypes)

#Visualize the Data

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(df['Date'],df['Calories'], marker='o')
plt.title('calories burned over time')
plt.xlabel('Date')
plt.ylabel("Calories")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(12,4))

sns.barplot(x="Workout_type", y="Calories",data=df, ax=axes[0])
axes[0].set_title("Average Calories by Workout Type")

sns.scatterplot(x="Duration", y="Calories", hue="Workout_type", data=df, s=90, ax=axes[1])
axes[1].set_title("Duration vs Calories")

plt.tight_layout()
plt.show()

In [ ]:
#Which weekday burns the most on average?
weekday_avg=df.groupby('Day_of_week')['Calories'].mean().sort_values(ascending=False)
weekday_avg

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(x=weekday_avg.index, y=weekday_avg.values)
plt.title('Average Calories burned by day of week')
plt.ylabel('Average Calories')
plt.xticks(rotation=30)
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
df.select_dtypes(include=np.number)

In [ ]:
numerical_cols=df.select_dtypes(include=np.number).columns
numerical_cols

In [ ]:
#Correlation Map
plt.figure(figsize=(8,6))
sns.heatmap(df[numerical_cols].corr(),
annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Between Numeric Columns")
plt.show()

#Save the Cleaned Dataset

In [ ]:
df.to_csv('cleaned_data.csv', index=False)
print("Saved the cleaned_data.csv")

from google.colab import files
files.download('cleaned_data.csv')

#Overall Summary

| Problem | How it was handled |
|---------|--------------------|
| Dates stored as quoted text | Stripped quotes, converted to real `datetime` |
| One date in `20201226` format | Handled automatically by `format='mixed'` |
| One missing date | Dropped that single row |
| One exact duplicate row | Removed with `drop_duplicates()` |
| Two missing `Calories` values | Filled with the median |
| `Duration` of 450 (typo for 45) | Corrected the value |
| `Pulse` higher than `Maxpulse` | Swapped the two values back |

**Result:** 32 rows → 30 clean rows, with 4 new derived columns (`Calories_per_minute`, `Intensity_percent`, `Day_of_Week`, `Workout_type`).